# Exploratory Data Analysis: USDA Branded Food Products
**Dataset:** usda_branded_sample_175k.csv (175,000 row random sample from 1,026,891 master records)

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

## 1) Load Dataset

In [ ]:
df = pd.read_csv("usda_branded_sample_175k.csv", low_memory=False)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(10)

## 2) Define Nutrient Columns

In [ ]:
NUTRIENTS = [
    "calories_per_serving",
    "protein_g_per_serving",
    "carbs_g_per_serving",
    "fat_g_per_serving",
    "fiber_g_per_serving",
    "sodium_mg_per_serving",
    "cholesterol_mg_per_serving",
    "satfat_g_per_serving",
    "transfat_g_per_serving",
    "total_sugar_g_per_serving",
    "added_sugar_g_per_serving",
]

# Confirm all present
for c in NUTRIENTS:
    assert c in df.columns, f"Missing: {c}"
print("All nutrient columns present.")

## 3) Summary Statistics

In [ ]:
summary = df[NUTRIENTS].describe(percentiles=[.05, .25, .5, .75, .95]).T.round(3)
summary['skewness'] = df[NUTRIENTS].skew().round(2)
summary

## 4) Missing Data Analysis

In [ ]:
miss_count = df[NUTRIENTS].isnull().sum()
miss_pct = (miss_count / len(df) * 100).round(2)
miss_df = pd.DataFrame({'missing_count': miss_count, 'missing_pct': miss_pct}).sort_values('missing_pct', ascending=False)
print(miss_df)

fig, ax = plt.subplots(figsize=(9, 5.5))
miss_sorted = miss_pct.sort_values(ascending=True)
colors = ['#C44E52' if p > 15 else '#4C72B0' for p in miss_sorted.values]
ax.barh(range(len(miss_sorted)), miss_sorted.values, color=colors, edgecolor='white')
ax.set_yticks(range(len(miss_sorted)))
labels = [c.replace('_per_serving','').replace('_g','(g)').replace('_mg','(mg)').replace('_',' ').title() for c in miss_sorted.index]
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Missing (%)')
ax.set_title('Figure 2. Missing Values by Nutrient Variable (n = 175,000)')
for i, val in enumerate(miss_sorted.values):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 5) Distribution of Calories

In [ ]:
cal = df['calories_per_serving'].dropna()
clip_val = cal.quantile(0.995)
cal_clip = cal[cal <= clip_val]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(cal_clip, bins=60, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(cal.median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: {cal.median():.0f} kcal')
ax.axvline(cal.mean(), color='orange', linestyle='--', linewidth=1.5, label=f'Mean: {cal.mean():.0f} kcal')
ax.set_xlabel('Calories per Serving (kcal)')
ax.set_ylabel('Frequency')
ax.set_title('Figure 1. Distribution of Calories per Serving')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Calorie stats: median={cal.median():.0f}, mean={cal.mean():.1f}, std={cal.std():.1f}, skewness={cal.skew():.2f}")

## 6) Correlation Analysis

In [ ]:
corr = df[NUTRIENTS].corr()

short_labels = ['Calories','Protein','Carbs','Fat','Fiber','Sodium',
                'Cholesterol','Sat Fat','Trans Fat','Total Sugar','Added Sugar']

fig, ax = plt.subplots(figsize=(10, 8.5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=short_labels, yticklabels=short_labels, ax=ax,
            vmin=-0.3, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.7}, annot_kws={'size': 8})
ax.set_title('Figure 3. Correlation Matrix of Nutrient Variables')
plt.tight_layout()
plt.show()

# Key correlations
print("Key correlations:")
print(f"  Fat vs Sat Fat: {corr.loc['fat_g_per_serving','satfat_g_per_serving']:.3f}")
print(f"  Calories vs Carbs: {corr.loc['calories_per_serving','carbs_g_per_serving']:.3f}")
print(f"  Calories vs Fat: {corr.loc['calories_per_serving','fat_g_per_serving']:.3f}")
print(f"  Total Sugar vs Added Sugar: {corr.loc['total_sugar_g_per_serving','added_sugar_g_per_serving']:.3f}")
print(f"  Total Sugar vs Carbs: {corr.loc['total_sugar_g_per_serving','carbs_g_per_serving']:.3f}")
print(f"  Carbs vs Fat: {corr.loc['carbs_g_per_serving','fat_g_per_serving']:.3f}")

## 7) Food Category Analysis

In [ ]:
print(f"Number of unique categories: {df['branded_food_category'].nunique()}")
print(f"Number of unique brands: {df['company_brand'].nunique()}")

# Top 15 categories by count
top_cats = df['branded_food_category'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(range(len(top_cats)), top_cats.values[::-1], color='#4C72B0', edgecolor='white')
ax.set_yticks(range(len(top_cats)))
ax.set_yticklabels(top_cats.index[::-1], fontsize=9)
ax.set_xlabel('Number of Products')
ax.set_title('Figure 4. Top 15 Food Categories by Product Count')
for i, v in enumerate(top_cats.values[::-1]):
    ax.text(v + 50, i, f'{v:,}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 8) Nutrient Boxplots by Category

In [ ]:
top_8_cats = df['branded_food_category'].value_counts().head(8).index.tolist()
df_top = df[df['branded_food_category'].isin(top_8_cats)].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
for i, (col, label, clip_hi) in enumerate([
    ('calories_per_serving', 'Calories (kcal)', 1000),
    ('fat_g_per_serving', 'Fat (g)', 50),
    ('sodium_mg_per_serving', 'Sodium (mg)', 2000)]):
    data_list, labels_list = [], []
    for cat in top_8_cats:
        vals = df_top.loc[df_top['branded_food_category']==cat, col].dropna()
        vals = vals[vals <= clip_hi]
        if len(vals) > 0:
            data_list.append(vals.values)
            short = cat.replace(' & ', '/').replace(', ', ',')
            if len(short) > 22: short = short[:19] + '...'
            labels_list.append(short)
    bp = axes[i].boxplot(data_list, vert=True, patch_artist=True, showfliers=False)
    for patch in bp['boxes']:
        patch.set_facecolor('#4C72B0')
        patch.set_alpha(0.6)
    axes[i].set_xticklabels(labels_list, rotation=50, ha='right', fontsize=7)
    axes[i].set_ylabel(label)
    axes[i].set_title(label)
fig.suptitle('Figure 5. Nutrient Distribution Across Top 8 Food Categories', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 9) Multivariate Scatter: Calories vs Fat vs Carbs

In [ ]:
df_plot = df.dropna(subset=['calories_per_serving','fat_g_per_serving','carbs_g_per_serving'])
df_plot = df_plot[(df_plot['calories_per_serving'] <= 1200) & (df_plot['fat_g_per_serving'] <= 80)]
if len(df_plot) > 8000:
    df_plot = df_plot.sample(8000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 5.5))
sc = ax.scatter(df_plot['fat_g_per_serving'], df_plot['calories_per_serving'],
                c=df_plot['carbs_g_per_serving'], cmap='viridis', alpha=0.35, s=12, edgecolors='none')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Carbs (g)')
ax.set_xlabel('Fat (g per Serving)')
ax.set_ylabel('Calories per Serving (kcal)')
ax.set_title('Figure 6. Calories vs. Fat, Colored by Carbohydrate Content')
plt.tight_layout()
plt.show()

## 10) Macronutrient Caloric Contribution

In [ ]:
df_m = df.dropna(subset=['calories_per_serving','protein_g_per_serving','carbs_g_per_serving','fat_g_per_serving']).copy()
df_m = df_m[df_m['calories_per_serving'] > 0]
df_m['carb_pct'] = df_m['carbs_g_per_serving'] * 4 / df_m['calories_per_serving'] * 100
df_m['fat_pct'] = df_m['fat_g_per_serving'] * 9 / df_m['calories_per_serving'] * 100
df_m['prot_pct'] = df_m['protein_g_per_serving'] * 4 / df_m['calories_per_serving'] * 100

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_m['carb_pct'].clip(0,150), bins=50, alpha=0.6, label='Carbs', color='#4C72B0')
ax.hist(df_m['fat_pct'].clip(0,150), bins=50, alpha=0.6, label='Fat', color='#C44E52')
ax.hist(df_m['prot_pct'].clip(0,150), bins=50, alpha=0.6, label='Protein', color='#55A868')
ax.set_xlabel('Percentage of Calories (%)')
ax.set_ylabel('Frequency')
ax.set_title('Figure 7. Macronutrient Contribution to Total Calories')
ax.legend()
ax.set_xlim(0, 120)
plt.tight_layout()
plt.show()

print(f"Protein: mean={df_m['prot_pct'].mean():.1f}%, median={df_m['prot_pct'].median():.1f}%")
print(f"Carbs:   mean={df_m['carb_pct'].mean():.1f}%, median={df_m['carb_pct'].median():.1f}%")
print(f"Fat:     mean={df_m['fat_pct'].mean():.1f}%, median={df_m['fat_pct'].median():.1f}%")

## 11) Sodium Analysis

In [ ]:
sod = df['sodium_mg_per_serving'].dropna()
sod_clip = sod[sod <= sod.quantile(0.99)]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(sod_clip, bins=60, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(460, color='red', linestyle='--', linewidth=2, label='FDA High Sodium (460 mg)')
ax.axvline(sod.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median: {sod.median():.0f} mg')
ax.set_xlabel('Sodium per Serving (mg)')
ax.set_ylabel('Frequency')
ax.set_title('Figure 8. Distribution of Sodium per Serving')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

pct_high = (df['sodium_mg_per_serving'] >= 460).mean()
print(f"Products exceeding 460mg sodium: {pct_high:.1%}")

# Top sodium categories
cat_sod = df.groupby('branded_food_category')['sodium_mg_per_serving'].agg(['median','count'])
cat_sod = cat_sod[cat_sod['count'] >= 50].sort_values('median', ascending=False)
print("\nTop 12 categories by median sodium:")
print(cat_sod.head(12))

# Bar chart
cat_sod_top = cat_sod.head(12).sort_values('median', ascending=True)
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = ['#C44E52' if m >= 460 else '#4C72B0' for m in cat_sod_top['median'].values]
ax.barh(range(len(cat_sod_top)), cat_sod_top['median'].values, color=colors, edgecolor='white')
ax.set_yticks(range(len(cat_sod_top)))
ax.set_yticklabels([c[:40] for c in cat_sod_top.index], fontsize=8)
ax.axvline(460, color='red', linestyle='--', linewidth=1.5, label='FDA High Sodium (460 mg)')
ax.set_xlabel('Median Sodium per Serving (mg)')
ax.set_title('Figure 10. Top 12 Categories by Median Sodium Content')
ax.legend(fontsize=9)
for i, v in enumerate(cat_sod_top['median'].values):
    ax.text(v + 50, i, f'{v:.0f} mg', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 12) Sugar Analysis

In [ ]:
sug = df['total_sugar_g_per_serving'].dropna()
sug_clip = sug[sug <= sug.quantile(0.99)]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(sug_clip, bins=60, color='#DD8452', edgecolor='white', alpha=0.85)
ax.axvline(sug.median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: {sug.median():.1f}g')
ax.axvline(sug.mean(), color='orange', linestyle='--', linewidth=1.5, label=f'Mean: {sug.mean():.1f}g')
ax.set_xlabel('Total Sugar per Serving (g)')
ax.set_ylabel('Frequency')
ax.set_title('Figure 11. Distribution of Total Sugar per Serving')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Total sugar: median={sug.median():.1f}g, mean={sug.mean():.1f}g")
print(f"Added sugar: median={df['added_sugar_g_per_serving'].median():.1f}g, mean={df['added_sugar_g_per_serving'].mean():.1f}g")

In [ ]:
# Added sugar vs total sugar
df_sug = df.dropna(subset=['total_sugar_g_per_serving','added_sugar_g_per_serving'])
df_sug = df_sug[(df_sug['total_sugar_g_per_serving'] <= 100) & (df_sug['added_sugar_g_per_serving'] <= 100)]
if len(df_sug) > 5000:
    df_sug_plot = df_sug.sample(5000, random_state=42)
else:
    df_sug_plot = df_sug

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(df_sug_plot['total_sugar_g_per_serving'], df_sug_plot['added_sugar_g_per_serving'],
           alpha=0.25, s=10, color='#DD8452', edgecolors='none')
ax.plot([0,100],[0,100], 'r--', linewidth=1, alpha=0.5, label='y = x (all sugar is added)')
ax.set_xlabel('Total Sugar per Serving (g)')
ax.set_ylabel('Added Sugar per Serving (g)')
ax.set_title('Figure 12. Added Sugar vs. Total Sugar per Serving')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Correlation (total vs added sugar): {df_sug['total_sugar_g_per_serving'].corr(df_sug['added_sugar_g_per_serving']):.3f}")

In [ ]:
# Top sugar categories
cat_sug = df.groupby('branded_food_category')['total_sugar_g_per_serving'].agg(['median','count'])
cat_sug = cat_sug[cat_sug['count'] >= 100].sort_values('median', ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(range(len(cat_sug)), cat_sug['median'].values, color='#DD8452', edgecolor='white')
ax.set_yticks(range(len(cat_sug)))
ax.set_yticklabels([c[:40] for c in cat_sug.index], fontsize=8)
ax.set_xlabel('Median Total Sugar per Serving (g)')
ax.set_title('Figure 13. Top 12 Categories by Median Sugar Content')
for i, v in enumerate(cat_sug['median'].values):
    ax.text(v + 0.5, i, f'{v:.1f}g', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 13) Trans Fat Analysis

In [ ]:
tf = df['transfat_g_per_serving'].dropna()
print(f"Trans fat: median={tf.median()}, mean={tf.mean():.3f}")
print(f"Products with trans fat > 0: {(tf > 0).sum():,} / {len(tf):,} = {(tf > 0).mean():.1%}")
print(f"Max trans fat: {tf.max():.1f}g")

# Of products with nonzero trans fat, what categories?
df_tf = df[df['transfat_g_per_serving'] > 0]
print(f"\nTop categories among products with trans fat > 0:")
print(df_tf['branded_food_category'].value_counts().head(10))

## 14) Serving Size Analysis

In [ ]:
df_ss = df.copy()
df_ss['unit_clean'] = df_ss['serving_size_unit'].replace({'GRM':'g','MLT':'ml','GM':'g'})

print("Serving size unit distribution:")
print(df_ss['unit_clean'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_g = df_ss[df_ss['unit_clean']=='g']
df_ml = df_ss[df_ss['unit_clean']=='ml']

axes[0].hist(df_g['serving_size'].clip(0,400), bins=50, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('Solid Products (grams)')
axes[0].set_xlabel('Serving Size (g)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df_g['serving_size'].median(), color='red', linestyle='--', label=f"Median: {df_g['serving_size'].median():.0f}g")
axes[0].legend(fontsize=9)

axes[1].hist(df_ml['serving_size'].clip(0,600), bins=40, color='#C44E52', edgecolor='white', alpha=0.85)
axes[1].set_title('Liquid Products (mL)')
axes[1].set_xlabel('Serving Size (mL)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(df_ml['serving_size'].median(), color='red', linestyle='--', label=f"Median: {df_ml['serving_size'].median():.0f}mL")
axes[1].legend(fontsize=9)

fig.suptitle('Figure 9. Serving Size Distribution by Measurement Unit', y=1.02)
plt.tight_layout()
plt.show()

## 15) Outlier Identification

In [ ]:
print("Top 5 extreme values by variable:\n")
for col in ['calories_per_serving', 'sodium_mg_per_serving', 'cholesterol_mg_per_serving']:
    print(f"=== {col} ===")
    print(df.nlargest(5, col)[['product','branded_food_category', col]].to_string())
    print()